In [1]:
# This is timport datarobot as dr
import pandas as pd
import numpy as np
from datetime import datetime
from scipy.stats import mode
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
pd.set_option('display.max_rows', 3000)

# 0 - Ingest Data

In [28]:
# Modulo1632_REC21_2023 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1632/968-Modulo1632/REC21_2024.csv", low_memory=False)
# Modulo1632_REC21_2023.info(verbose = True, show_counts = True)


# Build base directory (go up one level from current /02_scripts)
base_dir = os.path.dirname(os.getcwd())   # → c:\Users\linoc\OneDrive\Encoder\03_partos

# Build the path to the target CSV
csv_path_Modulo1633_REC41_2024_fil = os.path.join(base_dir,"03_bases_intermediarios", "Modulo1633_REC41_2024_fil.csv")
csv_path_Modulo1633_REC94_2024_fil = os.path.join(base_dir,"03_bases_intermediarios", "Modulo1633_REC94_2024_fil.csv")
# Build the path to the target CSV
csv_path_Modulo1631_REC91_2024 = os.path.join(base_dir,"01_raws", "2024", "968-Modulo1631", "968-Modulo1631", "REC91_2024.csv")
csv_path_Modulo1632_RE223132_2024= os.path.join(base_dir,"01_raws", "2024", "968-Modulo1632", "968-Modulo1632", "RE223132_2024.csv")
csv_path_Modulo1635_RE516171_2024 = os.path.join(base_dir,"01_raws", "2024", "968-Modulo1635", "968-Modulo1635", "RE516171_2024.csv")

# Load the dataset
Modulo1633_REC41_2024_fil = pd.read_csv(csv_path_Modulo1633_REC41_2024_fil, low_memory=False)
Modulo1633_REC94_2024_fil = pd.read_csv(csv_path_Modulo1633_REC94_2024_fil, low_memory=False)
Modulo1631_REC91_2024 = pd.read_csv(csv_path_Modulo1631_REC91_2024, low_memory=False)
Modulo1632_RE223132_2024 = pd.read_csv(csv_path_Modulo1632_RE223132_2024, low_memory=False)
Modulo1635_RE516171_2024 = pd.read_csv(csv_path_Modulo1635_RE516171_2024, low_memory=False)

Modulo1633_REC41_2024_fil.shape, Modulo1633_REC94_2024_fil.shape, Modulo1631_REC91_2024.shape, Modulo1632_RE223132_2024.shape, Modulo1635_RE516171_2024.shape

((17608, 467), (17608, 269), (37117, 343), (34252, 149), (34252, 84))

In [10]:
Modulo1632_RE223132_2024.CASEID.nunique()
key_variables =['CASEID']

# 1 Funtions

In [52]:
def save_file(df, output_file = "Modulo1633_REC41_2024_fil_clear.csv"):

    # File name only

    # Go one level up from the current working directory
    base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
    output_dir = os.path.join(base_dir, "03_bases_intermediarios")

    #Full path
    output_path = os.path.join(output_dir, output_file)

    # Save DataFrame
    df.to_csv(output_path, index=False, encoding="utf-8-sig")


def convert_objects_to_int64_safe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert object columns in a DataFrame to Int64 if possible.
    If conversion is not possible (non-integer strings), keep the column as object.
    """
    df_converted = df.copy()

    for col in df_converted.select_dtypes(include=["object"]).columns:
        s = df_converted[col].astype(str).str.strip()   # remove extra spaces
        s = s.mask(s == "", np.nan)                     # empty string -> NaN (no downcasting warning)
        
        # Try converting
        try:
            converted = pd.to_numeric(s, errors="raise").astype("Int64")
            df_converted[col] = converted
        except Exception:
            df_converted[col] = df_converted[col]  # keep original if fails
    
    return df_converted



def drop_null_and_list(df, exclude_list):
    """
    Drop columns that are entirely NaN or present in the exclude_list.

    Parameters:
        df (pd.DataFrame): Input DataFrame.
        exclude_list (list): List of column names to exclude.

    Returns:
        pd.DataFrame: DataFrame with specified columns removed.
    """
    # Drop all-null columns
    df_filtered = df.dropna(axis=1, how="all")
    
    # Drop columns from exclude_list
    df_filtered = df_filtered.drop(columns=[col for col in exclude_list if col in df_filtered.columns], errors="ignore")
    
    return df_filtered



def drop_high_null_columns(df, threshold=0.45):
    """
    Drop columns from a DataFrame where the percentage of null values 
    is greater than the given threshold.

    Parameters:
        df (pd.DataFrame): Input DataFrame.
        threshold (float): Proportion threshold (default 0.45 means 45%).

    Returns:
        pd.DataFrame: DataFrame with high-null columns removed.
    """
    # Calculate the null ratio for each column
    null_ratio = df.isnull().mean()

    # Keep only columns with null ratio <= threshold
    df_filtered = df.loc[:, null_ratio <= threshold]

    return df_filtered


# 2 Data Wrangling

# 2.1 selection variable type

In [ ]:
def categorize_columns(df, key_variables):
    numeric_cols = []
    categorical_cols = []
    dummy_cols = []

    features = df.columns.to_list()
    #features.remove(target_variable)
    features =[item for item in features  if item not in key_variables]

    # Iterate over each column in DataFrame
    for col in features:
        if pd.api.types.is_numeric_dtype(df[col]):
            # Check if the column is a dummy variable
            unique_values = pd.Series(df[col].dropna().unique())
            
            if unique_values.isin([0, 1]).all() and unique_values.size <= 2:
                dummy_cols.append(col)
            else:
                numeric_cols.append(col)
        elif pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
            categorical_cols.append(col)
    
    print(len(numeric_cols)),print(len(categorical_cols)), print(len(dummy_cols)) 
    
    return numeric_cols, categorical_cols, dummy_cols


##  2.2 - Treatment of Numerical Data: Outlier, fill Missing Values and Normalize

In [5]:
# def fill_missing_with_median_coding(df):
#     """
#     Drops values greater than the 99th percentile, fills missing values in each column 
#     of the DataFrame with the median of that column, and applies MinMax scaling to each 
#     numeric column.

#     Parameters:
#     df (pd.DataFrame): The DataFrame with missing values.

#     Returns:
#     pd.DataFrame: A DataFrame with values above 99th percentile dropped, missing values 
#     filled, and numeric columns scaled from 0 to 1.
#     """

#     # Calculate the 99th percentile for each column and drop rows with any value above it
    
#     for column in df.columns.to_list():
#         percentile_99 = df[column].quantile(0.99)
#         max_value_below_99 = df[df[column] < percentile_99][column].max()
#         df[column] = df[column].apply(lambda x: max_value_below_99 if x > percentile_99 else x)
        
#     df_filled = df.copy()
    
#     for column in df_filled.columns:
#         median_value = df_filled[column].median()
#         #df_filled[column].fillna(median_value, inplace=True)
#         df_filled[column] = df_filled[column].fillna(median_value)

#         scaler = MinMaxScaler()
#         df_filled[column] = scaler.fit_transform(df_filled[[column]]) 

#     print(df_filled.shape)
    
#     return df_filled



def fill_missing_with_median_coding(df):
    """
    1. Cap values above 99th percentile to the max below that threshold.
    2. Fill missing values with column median.
    3. Scale each column between 0 and 1 with MinMaxScaler.
    
    Parameters:
    ----------
    df : pd.DataFrame
        Input DataFrame with numeric columns only.

    Returns:
    -------
    pd.DataFrame
        Transformed DataFrame with capped, imputed, and scaled values.
    """
    df_filled = df.copy()

    for column in df_filled.columns:
        # Step 1: cap at 99th percentile
        percentile_99 = df_filled[column].quantile(0.99)
        max_value_below_99 = df_filled.loc[df_filled[column] < percentile_99, column].max()
        df_filled.loc[df_filled[column] > percentile_99, column] = max_value_below_99

        # Step 2: fill missing with median
        median_value = df_filled[column].median()
        df_filled[column] = df_filled[column].fillna(median_value)

        # Step 3: scale between 0 and 1
        scaler = MinMaxScaler()
        df_filled[[column]] = scaler.fit_transform(df_filled[[column]])

    print("Shape after transformation:", df_filled.shape)
    return df_filled


## 2.3 - Treatment of categorical  Data: Fill Missing Values and encoding

In [6]:
# function missing values 
def fill_missing_with_mode(df):
    """
    Fills missing values in each column of the DataFrame with the median of that column.

    Parameters:
    df (pd.DataFrame): The DataFrame with missing values.

    Returns:
    pd.DataFrame: A DataFrame with missing values filled with the median of their respective columns.
    """
    df_filled = df.copy()
    
    for column in df_filled.columns:
        mode_value = df_filled[column].mode()[0]
        #df_filled[column].fillna(median_value, inplace=True)
        df_filled[column] = df_filled[column].fillna(mode_value)
    
    return df_filled


# function encoding 
def target_encode(df, categorical_columns, target_column):
    # Create a copy of the DataFrame to avoid modifying the original data
    df_encoded = df.copy()
    
    # For each categorical feature, perform target encoding
    for column in categorical_columns:
        # Create a dictionary of category: average target
        target_means = df.groupby(column)[target_column].mean()
        # Map the categorical features to these target averages
        df_encoded[column] = df[column].map(target_means)
    
    df_encoded.drop(columns = target_column, inplace = True)
        
    return df_encoded

## 2.4 Flag

In [7]:
#df_load_dummy_cols =  fill_missing_with_mode(df_load_join[dummy_cols])

# 3 - DBS 

## 3.1 csv_path_Modulo1633_REC41_2024_fil

In [9]:
key_variables =['CASEID']
numeric_cols, categorical_cols, dummy_cols = categorize_columns (Modulo1633_REC41_2024_fil, key_variables)

143
0
323


In [10]:
df_numeric_cols = fill_missing_with_median_coding(Modulo1633_REC41_2024_fil[numeric_cols])

C:\Users\linoc\AppData\Local\Temp\ipykernel_60672\3211958872.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].apply(lambda x: max_value_below_99 if x > percentile_99 else x)
C:\Users\linoc\AppData\Local\Temp\ipykernel_60672\3211958872.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].apply(lambda x: max_value_below_99 if x > percentile_99 else x)
C:\Users\linoc\AppData\Local\Temp\ipykernel_60672\3211958872.py:20: SettingWithCopyWarning: 
A value is tryi

(17608, 143)


C:\Users\linoc\AppData\Local\Temp\ipykernel_60672\3211958872.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filled[column].fillna(median_value, inplace=True)
C:\Users\linoc\AppData\Local\Temp\ipykernel_60672\3211958872.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [11]:
#df_categorical_cols = fill_missing_with_mode(Modulo1633_REC41_2024_fil[categorical_cols])
#df_categorical_cols = target_encode((pd.concat([df_categorical_cols, Modulo1633_REC41_2024_fil['SURVEY_TARGET_SSPIVISN40']], axis=1)), categorical_cols, 'SURVEY_TARGET_SSPIVISN40')

In [12]:
df_load_dummy_cols =  fill_missing_with_mode(Modulo1633_REC41_2024_fil[dummy_cols])

In [16]:
Modulo1633_REC41_2024_fil_clear = pd.concat([df_numeric_cols, df_load_dummy_cols, Modulo1633_REC41_2024_fil[key_variables]], axis=1)
Modulo1633_REC41_2024_fil_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 467 columns):
 #    Column     Non-Null Count  Dtype  
---   ------     --------------  -----  
 0    M3A        17608 non-null  float64
 1    M3B        17608 non-null  float64
 2    M3C        17608 non-null  float64
 3    M3D        17608 non-null  float64
 4    M3G        17608 non-null  float64
 5    M3H        17608 non-null  float64
 6    M3K        17608 non-null  float64
 7    M3N        17608 non-null  float64
 8    M17        17608 non-null  float64
 9    M66        17608 non-null  float64
 10   ID1_2024   17608 non-null  float64
 11   M1_<NA>    17608 non-null  float64
 12   M1A_<NA>   17608 non-null  float64
 13   M1B_<NA>   17608 non-null  float64
 14   M1D_<NA>   17608 non-null  float64
 15   M5_0       17608 non-null  float64
 16   M5_1       17608 non-null  float64
 17   M5_10      17608 non-null  float64
 18   M5_11      17608 non-null  float64
 19   M5_12      17608 non-nu

In [17]:
# File name only
output_file = "Modulo1633_REC41_2024_fil_clear.csv"

# Go one level up from the current working directory
base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
output_dir = os.path.join(base_dir, "03_bases_intermediarios")

#Full path
output_path = os.path.join(output_dir, output_file)

# Save DataFrame
Modulo1633_REC41_2024_fil.to_csv(output_path, index=False, encoding="utf-8-sig")

## 3.2 Modulo1633_REC94_2024_fil

In [8]:
Modulo1633_REC94_2024_fil.info(verbose = True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 269 columns):
 #    Column         Non-Null Count  Dtype  
---   ------         --------------  -----  
 0    CASEID         17608 non-null  object 
 1    S413           17608 non-null  int64  
 2    S426E          17608 non-null  int64  
 3    S426GA         17608 non-null  int64  
 4    S426GB         17608 non-null  int64  
 5    S426GC         17608 non-null  int64  
 6    S426GD         17608 non-null  int64  
 7    S426GE         17608 non-null  int64  
 8    S430D          17608 non-null  int64  
 9    S427DA         17608 non-null  int64  
 10   S427DB         17608 non-null  int64  
 11   S427DC         17608 non-null  int64  
 12   S427DD         17608 non-null  int64  
 13   S427DE         17608 non-null  int64  
 14   S427DF         17608 non-null  int64  
 15   S427DG         17608 non-null  int64  
 16   S427F          17608 non-null  int64  
 17   S436C          17608 non-null

In [11]:
numeric_cols_REC94, categorical_cols_REC94, dummy_cols_REC94 = categorize_columns (Modulo1633_REC94_2024_fil, key_variables)

92
0
176


In [14]:
df_numeric_cols_REC94 = fill_missing_with_median_coding(Modulo1633_REC94_2024_fil[numeric_cols_REC94])
df_numeric_cols_REC94.info(verbose = True, show_counts = True)

Shape after transformation: (17608, 92)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 92 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   S426E          17608 non-null  float64
 1   S430D          17608 non-null  float64
 2   S436C          17608 non-null  float64
 3   S441           17608 non-null  float64
 4   ID1_2024       17608 non-null  float64
 5   S410B_<NA>     17608 non-null  float64
 6   S411B_<NA>     17608 non-null  float64
 7   S411F_<NA>     17608 non-null  float64
 8   S411G_<NA>     17608 non-null  float64
 9   S411H_<NA>     17608 non-null  float64
 10  S411I_<NA>     17608 non-null  float64
 11  S411J_<NA>     17608 non-null  float64
 12  S411K_<NA>     17608 non-null  float64
 13  S411L_<NA>     17608 non-null  float64
 14  S411BA_<NA>    17608 non-null  float64
 15  S411CA_<NA>    17608 non-null  float64
 16  S411DA_<NA>    17608 non-null  float64
 17  S411EA_<NA

In [13]:
df_load_dummy_cols_REC94 =  fill_missing_with_mode(Modulo1633_REC94_2024_fil[dummy_cols_REC94])
df_load_dummy_cols_REC94.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 176 columns):
 #    Column      Non-Null Count  Dtype
---   ------      --------------  -----
 0    S413        17608 non-null  int64
 1    S426GA      17608 non-null  int64
 2    S426GB      17608 non-null  int64
 3    S426GC      17608 non-null  int64
 4    S426GD      17608 non-null  int64
 5    S426GE      17608 non-null  int64
 6    S427DA      17608 non-null  int64
 7    S427DB      17608 non-null  int64
 8    S427DC      17608 non-null  int64
 9    S427DD      17608 non-null  int64
 10   S427DE      17608 non-null  int64
 11   S427DF      17608 non-null  int64
 12   S427DG      17608 non-null  int64
 13   S427F       17608 non-null  int64
 14   IDX94_1     17608 non-null  int64
 15   IDX94_2     17608 non-null  int64
 16   IDX94_3     17608 non-null  int64
 17   IDX94_4     17608 non-null  int64
 18   S410B_1     17608 non-null  int64
 19   S410B_2     17608 non-null  int64
 20   S410

In [15]:
Modulo1633_REC94_2024_fil_clear = pd.concat([df_numeric_cols_REC94, df_load_dummy_cols_REC94, Modulo1633_REC94_2024_fil[key_variables]], axis=1)
Modulo1633_REC94_2024_fil_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 269 columns):
 #    Column         Non-Null Count  Dtype  
---   ------         --------------  -----  
 0    S426E          17608 non-null  float64
 1    S430D          17608 non-null  float64
 2    S436C          17608 non-null  float64
 3    S441           17608 non-null  float64
 4    ID1_2024       17608 non-null  float64
 5    S410B_<NA>     17608 non-null  float64
 6    S411B_<NA>     17608 non-null  float64
 7    S411F_<NA>     17608 non-null  float64
 8    S411G_<NA>     17608 non-null  float64
 9    S411H_<NA>     17608 non-null  float64
 10   S411I_<NA>     17608 non-null  float64
 11   S411J_<NA>     17608 non-null  float64
 12   S411K_<NA>     17608 non-null  float64
 13   S411L_<NA>     17608 non-null  float64
 14   S411BA_<NA>    17608 non-null  float64
 15   S411CA_<NA>    17608 non-null  float64
 16   S411DA_<NA>    17608 non-null  float64
 17   S411EA_<NA>    17608 non-null

In [17]:
save_file(Modulo1633_REC94_2024_fil_clear, output_file = "Modulo1633_REC94_2024_fil_clear.csv")


## Modulo1631_REC91_2024

In [29]:
Modulo1631_REC91_2024.info(verbose= True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 343 columns):
 #    Column    Non-Null Count  Dtype 
---   ------    --------------  ----- 
 0    ID1       37117 non-null  int64 
 1    CASEID    37117 non-null  object
 2    SVER      37117 non-null  int64 
 3    SREGION   37117 non-null  int64 
 4    SSEMES    37117 non-null  int64 
 5    SPROVIN   37117 non-null  int64 
 6    SDISTRI   37117 non-null  int64 
 7    S108N     37117 non-null  object
 8    S108Y     37117 non-null  object
 9    S108G     37117 non-null  object
 10   S111      37117 non-null  object
 11   S112      37117 non-null  object
 12   S119      37117 non-null  object
 13   S119NA    37117 non-null  object
 14   S119NB    37117 non-null  object
 15   S119D     37117 non-null  object
 16   S229A     37117 non-null  object
 17   S229B     37117 non-null  object
 18   S229C     37117 non-null  object
 19   S229D     37117 non-null  object
 20   S229E     37117 non-null  

In [53]:
Modulo1631_REC91_2024_M = convert_objects_to_int64_safe(Modulo1631_REC91_2024)
Modulo1631_REC91_2024_M = Modulo1631_REC91_2024_M.dropna(axis=1, how="all")
Modulo1631_REC91_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 299 columns):
 #    Column    Non-Null Count  Dtype 
---   ------    --------------  ----- 
 0    ID1       37117 non-null  int64 
 1    CASEID    37117 non-null  object
 2    SVER      37117 non-null  int64 
 3    SREGION   37117 non-null  int64 
 4    SSEMES    37117 non-null  int64 
 5    SPROVIN   37117 non-null  int64 
 6    SDISTRI   37117 non-null  int64 
 7    S108N     34252 non-null  Int64 
 8    S108Y     33910 non-null  Int64 
 9    S108G     6326 non-null   Int64 
 10   S111      12058 non-null  Int64 
 11   S112      4623 non-null   Int64 
 12   S119      34252 non-null  Int64 
 13   S119NA    27810 non-null  Int64 
 14   S119NB    27810 non-null  Int64 
 15   S119D     34252 non-null  Int64 
 16   S229A     723 non-null    Int64 
 17   S229B     723 non-null    Int64 
 18   S229C     723 non-null    Int64 
 19   S229D     723 non-null    Int64 
 20   S229E     723 non-null    

In [54]:
Modulo1631_REC91_2024_M = drop_high_null_columns(Modulo1631_REC91_2024_M , threshold=0.45)
Modulo1631_REC91_2024_M.info(verbose = True, show_counts = True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 132 columns):
 #    Column   Non-Null Count  Dtype 
---   ------   --------------  ----- 
 0    ID1      37117 non-null  int64 
 1    CASEID   37117 non-null  object
 2    SVER     37117 non-null  int64 
 3    SREGION  37117 non-null  int64 
 4    SSEMES   37117 non-null  int64 
 5    SPROVIN  37117 non-null  int64 
 6    SDISTRI  37117 non-null  int64 
 7    S108N    34252 non-null  Int64 
 8    S108Y    33910 non-null  Int64 
 9    S119     34252 non-null  Int64 
 10   S119NA   27810 non-null  Int64 
 11   S119NB   27810 non-null  Int64 
 12   S119D    34252 non-null  Int64 
 13   S239A    34252 non-null  Int64 
 14   S239B    34252 non-null  Int64 
 15   S239C    34252 non-null  Int64 
 16   S239D    34252 non-null  Int64 
 17   S239E    34252 non-null  Int64 
 18   S239F    34252 non-null  Int64 
 19   S239X    34252 non-null  Int64 
 20   S489C    30623 non-null  Int64 
 21   S489D    3

In [55]:
numeric_cols_REC91, categorical_cols_REC91, dummy_cols_REC91 = categorize_columns (Modulo1631_REC91_2024_M, key_variables)

36
0
95


In [56]:
df_numeric_cols_REC91 = fill_missing_with_median_coding(Modulo1631_REC91_2024_M[numeric_cols_REC91])
df_numeric_cols_REC91.info(verbose = True, show_counts = True)

Shape after transformation: (37117, 36)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 36 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID1      37117 non-null  float64
 1   SREGION  37117 non-null  float64
 2   SSEMES   37117 non-null  float64
 3   SPROVIN  37117 non-null  float64
 4   SDISTRI  37117 non-null  float64
 5   S108N    37117 non-null  float64
 6   S108Y    37117 non-null  float64
 7   S119     37117 non-null  float64
 8   S119NA   37117 non-null  float64
 9   S119NB   37117 non-null  float64
 10  S119D    37117 non-null  float64
 11  S489C    37117 non-null  float64
 12  S489D    37117 non-null  float64
 13  S490     37117 non-null  float64
 14  S704N    37117 non-null  float64
 15  S704Y    37117 non-null  float64
 16  S802     37117 non-null  float64
 17  S802D    37117 non-null  float64
 18  S802E    37117 non-null  float64
 19  S802F    37117 non-null  float64
 20  S802I    3

In [59]:
df_load_dummy_cols_REC91 =  fill_missing_with_mode(Modulo1631_REC91_2024_M[dummy_cols_REC91])
df_load_dummy_cols_REC91.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 95 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   SVER    37117 non-null  int64
 1   S239A   37117 non-null  Int64
 2   S239B   37117 non-null  Int64
 3   S239C   37117 non-null  Int64
 4   S239D   37117 non-null  Int64
 5   S239E   37117 non-null  Int64
 6   S239F   37117 non-null  Int64
 7   S239X   37117 non-null  Int64
 8   S490AA  37117 non-null  Int64
 9   S490AB  37117 non-null  Int64
 10  S490AC  37117 non-null  Int64
 11  S490AD  37117 non-null  Int64
 12  S490AE  37117 non-null  Int64
 13  S490AF  37117 non-null  Int64
 14  S490AG  37117 non-null  Int64
 15  S490AX  37117 non-null  Int64
 16  S490BA  37117 non-null  Int64
 17  S490BB  37117 non-null  Int64
 18  S490BC  37117 non-null  Int64
 19  S490BD  37117 non-null  Int64
 20  S490BX  37117 non-null  Int64
 21  S500A   37117 non-null  Int64
 22  S500B   37117 non-null  Int64
 23  S500C   371

In [60]:
Modulo1631_REC91_2024_M_clear = pd.concat([df_numeric_cols_REC91, df_load_dummy_cols_REC91, Modulo1631_REC91_2024_M[key_variables]], axis=1)
Modulo1631_REC91_2024_M_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 132 columns):
 #    Column   Non-Null Count  Dtype  
---   ------   --------------  -----  
 0    ID1      37117 non-null  float64
 1    SREGION  37117 non-null  float64
 2    SSEMES   37117 non-null  float64
 3    SPROVIN  37117 non-null  float64
 4    SDISTRI  37117 non-null  float64
 5    S108N    37117 non-null  float64
 6    S108Y    37117 non-null  float64
 7    S119     37117 non-null  float64
 8    S119NA   37117 non-null  float64
 9    S119NB   37117 non-null  float64
 10   S119D    37117 non-null  float64
 11   S489C    37117 non-null  float64
 12   S489D    37117 non-null  float64
 13   S490     37117 non-null  float64
 14   S704N    37117 non-null  float64
 15   S704Y    37117 non-null  float64
 16   S802     37117 non-null  float64
 17   S802D    37117 non-null  float64
 18   S802E    37117 non-null  float64
 19   S802F    37117 non-null  float64
 20   S802I    37117 non-null  f

In [61]:
save_file(Modulo1631_REC91_2024_M_clear, output_file = "Modulo1631_REC91_2024_M_clear.csv")

## Modulo1632_RE223132_2024

In [62]:
Modulo1632_RE223132_2024.info(verbose = True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 149 columns):
 #    Column     Non-Null Count  Dtype 
---   ------     --------------  ----- 
 0    ID1        34252 non-null  int64 
 1    CASEID     34252 non-null  object
 2    V201       34252 non-null  int64 
 3    V202       34252 non-null  int64 
 4    V203       34252 non-null  int64 
 5    V204       34252 non-null  int64 
 6    V205       34252 non-null  int64 
 7    V206       34252 non-null  int64 
 8    V207       34252 non-null  int64 
 9    V208       34252 non-null  int64 
 10   V209       34252 non-null  int64 
 11   V210       34252 non-null  int64 
 12   V211       34252 non-null  object
 13   V212       34252 non-null  object
 14   V213       34252 non-null  int64 
 15   V214       34252 non-null  object
 16   V215       34252 non-null  int64 
 17   V216       34252 non-null  int64 
 18   V217       34252 non-null  int64 
 19   V218       34252 non-null  int64 
 20   V219

In [63]:
Modulo1632_RE223132_2024_M = convert_objects_to_int64_safe(Modulo1632_RE223132_2024)
Modulo1632_RE223132_2024_M = Modulo1632_RE223132_2024_M.dropna(axis=1, how="all")
Modulo1632_RE223132_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 135 columns):
 #    Column     Non-Null Count  Dtype 
---   ------     --------------  ----- 
 0    ID1        34252 non-null  int64 
 1    CASEID     34252 non-null  object
 2    V201       34252 non-null  int64 
 3    V202       34252 non-null  int64 
 4    V203       34252 non-null  int64 
 5    V204       34252 non-null  int64 
 6    V205       34252 non-null  int64 
 7    V206       34252 non-null  int64 
 8    V207       34252 non-null  int64 
 9    V208       34252 non-null  int64 
 10   V209       34252 non-null  int64 
 11   V210       34252 non-null  int64 
 12   V211       24627 non-null  Int64 
 13   V212       24627 non-null  Int64 
 14   V213       34252 non-null  int64 
 15   V214       723 non-null    Int64 
 16   V215       34252 non-null  int64 
 17   V216       34252 non-null  int64 
 18   V217       34252 non-null  int64 
 19   V218       34252 non-null  int64 
 20   V219

In [64]:
Modulo1632_RE223132_2024_M = drop_high_null_columns(Modulo1632_RE223132_2024_M , threshold=0.45)
Modulo1632_RE223132_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 65 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID1        34252 non-null  int64 
 1   CASEID     34252 non-null  object
 2   V201       34252 non-null  int64 
 3   V202       34252 non-null  int64 
 4   V203       34252 non-null  int64 
 5   V204       34252 non-null  int64 
 6   V205       34252 non-null  int64 
 7   V206       34252 non-null  int64 
 8   V207       34252 non-null  int64 
 9   V208       34252 non-null  int64 
 10  V209       34252 non-null  int64 
 11  V210       34252 non-null  int64 
 12  V211       24627 non-null  Int64 
 13  V212       24627 non-null  Int64 
 14  V213       34252 non-null  int64 
 15  V215       34252 non-null  int64 
 16  V216       34252 non-null  int64 
 17  V217       34252 non-null  int64 
 18  V218       34252 non-null  int64 
 19  V219       34252 non-null  int64 
 20  V220       34252 non-null  i

In [65]:
numeric_cols_RE223132, categorical_cols_RE223132, dummy_cols_RE223132 = categorize_columns (Modulo1632_RE223132_2024_M, key_variables)

41
0
23


In [66]:
df_numeric_cols_RE223132 = fill_missing_with_median_coding(Modulo1632_RE223132_2024_M[numeric_cols_RE223132])
df_numeric_cols_RE223132.info(verbose = True, show_counts = True)

Shape after transformation: (34252, 41)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 41 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID1        34252 non-null  float64
 1   V201       34252 non-null  float64
 2   V202       34252 non-null  float64
 3   V203       34252 non-null  float64
 4   V204       34252 non-null  float64
 5   V205       34252 non-null  float64
 6   V206       34252 non-null  float64
 7   V207       34252 non-null  float64
 8   V208       34252 non-null  float64
 9   V209       34252 non-null  float64
 10  V210       34252 non-null  float64
 11  V211       34252 non-null  float64
 12  V212       34252 non-null  float64
 13  V215       34252 non-null  float64
 14  V217       34252 non-null  float64
 15  V218       34252 non-null  float64
 16  V219       34252 non-null  float64
 17  V220       34252 non-null  float64
 18  V221       34252 non-null  float64
 19  V222  

In [67]:
df_load_dummy_cols_RE223132 =  fill_missing_with_mode(Modulo1632_RE223132_2024_M[dummy_cols_RE223132])
df_load_dummy_cols_RE223132.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 23 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   V213     34252 non-null  int64
 1   V216     34252 non-null  int64
 2   V228     34252 non-null  int64
 3   V384A    34252 non-null  int64
 4   V384B    34252 non-null  int64
 5   V384C    34252 non-null  int64
 6   V393     34252 non-null  Int64
 7   V394     34252 non-null  Int64
 8   V305_01  34252 non-null  int64
 9   V305_02  34252 non-null  int64
 10  V305_03  34252 non-null  int64
 11  V305_05  34252 non-null  int64
 12  V305_06  34252 non-null  int64
 13  V305_07  34252 non-null  int64
 14  V305_08  34252 non-null  int64
 15  V305_09  34252 non-null  int64
 16  V305_10  34252 non-null  int64
 17  V305_11  34252 non-null  int64
 18  V305_13  34252 non-null  int64
 19  V305_14  34252 non-null  int64
 20  V305_15  34252 non-null  int64
 21  V305_16  34252 non-null  int64
 22  V307_03  34252 non-nul

In [68]:
Modulo1632_RE223132_2024_M_clear = pd.concat([df_numeric_cols_RE223132, df_load_dummy_cols_RE223132, Modulo1632_RE223132_2024_M[key_variables]], axis=1)
Modulo1632_RE223132_2024_M_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 65 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID1        34252 non-null  float64
 1   V201       34252 non-null  float64
 2   V202       34252 non-null  float64
 3   V203       34252 non-null  float64
 4   V204       34252 non-null  float64
 5   V205       34252 non-null  float64
 6   V206       34252 non-null  float64
 7   V207       34252 non-null  float64
 8   V208       34252 non-null  float64
 9   V209       34252 non-null  float64
 10  V210       34252 non-null  float64
 11  V211       34252 non-null  float64
 12  V212       34252 non-null  float64
 13  V215       34252 non-null  float64
 14  V217       34252 non-null  float64
 15  V218       34252 non-null  float64
 16  V219       34252 non-null  float64
 17  V220       34252 non-null  float64
 18  V221       34252 non-null  float64
 19  V222       34252 non-null  float64
 20  V224  

In [69]:
save_file(Modulo1632_RE223132_2024_M_clear, output_file = "Modulo1632_RE223132_2024_M_clear.csv")

## Modulo1635_RE516171_2024

In [70]:
Modulo1635_RE516171_2024.info(verbose= True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 84 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID1     34252 non-null  int64 
 1   CASEID  34252 non-null  object
 2   V501    34252 non-null  int64 
 3   V502    34252 non-null  int64 
 4   V503    34252 non-null  object
 5   V504    34252 non-null  object
 6   V505    34252 non-null  object
 7   V506    34252 non-null  object
 8   V507    34252 non-null  object
 9   V508    34252 non-null  object
 10  V509    34252 non-null  object
 11  V510    34252 non-null  object
 12  V511    34252 non-null  object
 13  V512    34252 non-null  object
 14  V513    34252 non-null  int64 
 15  V525    34252 non-null  int64 
 16  V527    34252 non-null  object
 17  V528    34252 non-null  object
 18  V529    34252 non-null  object
 19  V530    34252 non-null  object
 20  V531    34252 non-null  int64 
 21  V532    34252 non-null  int64 
 22  V535    34252 non-null

In [71]:
Modulo1635_RE516171_2024_M = convert_objects_to_int64_safe(Modulo1635_RE516171_2024)
Modulo1635_RE516171_2024_M = Modulo1635_RE516171_2024_M.dropna(axis=1, how="all")
Modulo1635_RE516171_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 76 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID1     34252 non-null  int64 
 1   CASEID  34252 non-null  object
 2   V501    34252 non-null  int64 
 3   V502    34252 non-null  int64 
 4   V503    24189 non-null  Int64 
 5   V504    19780 non-null  Int64 
 6   V507    24189 non-null  Int64 
 7   V508    24189 non-null  Int64 
 8   V509    24189 non-null  Int64 
 9   V510    24189 non-null  Int64 
 10  V511    24189 non-null  Int64 
 11  V512    24189 non-null  Int64 
 12  V513    34252 non-null  int64 
 13  V525    34252 non-null  int64 
 14  V527    27705 non-null  Int64 
 15  V528    27705 non-null  Int64 
 16  V529    27705 non-null  Int64 
 17  V530    27705 non-null  Int64 
 18  V531    34252 non-null  int64 
 19  V532    34252 non-null  int64 
 20  V535    14472 non-null  Int64 
 21  V536    34252 non-null  int64 
 22  V537    8581 non-null 

In [72]:
Modulo1635_RE516171_2024_M = drop_high_null_columns(Modulo1635_RE516171_2024_M , threshold=0.45)
Modulo1635_RE516171_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 63 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID1     34252 non-null  int64 
 1   CASEID  34252 non-null  object
 2   V501    34252 non-null  int64 
 3   V502    34252 non-null  int64 
 4   V503    24189 non-null  Int64 
 5   V504    19780 non-null  Int64 
 6   V507    24189 non-null  Int64 
 7   V508    24189 non-null  Int64 
 8   V509    24189 non-null  Int64 
 9   V510    24189 non-null  Int64 
 10  V511    24189 non-null  Int64 
 11  V512    24189 non-null  Int64 
 12  V513    34252 non-null  int64 
 13  V525    34252 non-null  int64 
 14  V527    27705 non-null  Int64 
 15  V528    27705 non-null  Int64 
 16  V529    27705 non-null  Int64 
 17  V530    27705 non-null  Int64 
 18  V531    34252 non-null  int64 
 19  V532    34252 non-null  int64 
 20  V536    34252 non-null  int64 
 21  V602    34252 non-null  int64 
 22  V605    34252 non-null

In [73]:
numeric_cols_RE516171, categorical_cols_RE516171, dummy_cols_RE516171 = categorize_columns (Modulo1635_RE516171_2024_M, key_variables)

61
0
1


In [74]:
df_numeric_cols_RE516171 = fill_missing_with_median_coding(Modulo1635_RE516171_2024_M[numeric_cols_RE516171])
df_numeric_cols_RE516171.info(verbose = True, show_counts = True)

Shape after transformation: (34252, 61)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 61 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID1     34252 non-null  float64
 1   V501    34252 non-null  float64
 2   V502    34252 non-null  float64
 3   V503    34252 non-null  float64
 4   V504    34252 non-null  float64
 5   V507    34252 non-null  float64
 6   V508    34252 non-null  float64
 7   V509    34252 non-null  float64
 8   V510    34252 non-null  float64
 9   V511    34252 non-null  float64
 10  V512    34252 non-null  float64
 11  V513    34252 non-null  float64
 12  V525    34252 non-null  float64
 13  V527    34252 non-null  float64
 14  V528    34252 non-null  float64
 15  V529    34252 non-null  float64
 16  V530    34252 non-null  float64
 17  V531    34252 non-null  float64
 18  V532    34252 non-null  float64
 19  V536    34252 non-null  float64
 20  V602    34252 non-null  float64


In [75]:
df_load_dummy_cols_RE516171 =  fill_missing_with_mode(Modulo1635_RE516171_2024_M[dummy_cols_RE516171])
df_load_dummy_cols_RE516171.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   V714    34252 non-null  int64
dtypes: int64(1)
memory usage: 267.7 KB


In [76]:
Modulo1635_RE516171_2024_M_clear = pd.concat([df_numeric_cols_RE516171, df_load_dummy_cols_RE516171, Modulo1635_RE516171_2024_M[key_variables]], axis=1)
Modulo1635_RE516171_2024_M_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 63 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID1     34252 non-null  float64
 1   V501    34252 non-null  float64
 2   V502    34252 non-null  float64
 3   V503    34252 non-null  float64
 4   V504    34252 non-null  float64
 5   V507    34252 non-null  float64
 6   V508    34252 non-null  float64
 7   V509    34252 non-null  float64
 8   V510    34252 non-null  float64
 9   V511    34252 non-null  float64
 10  V512    34252 non-null  float64
 11  V513    34252 non-null  float64
 12  V525    34252 non-null  float64
 13  V527    34252 non-null  float64
 14  V528    34252 non-null  float64
 15  V529    34252 non-null  float64
 16  V530    34252 non-null  float64
 17  V531    34252 non-null  float64
 18  V532    34252 non-null  float64
 19  V536    34252 non-null  float64
 20  V602    34252 non-null  float64
 21  V605    34252 non-null  float64
 22

In [77]:
save_file(Modulo1635_RE516171_2024_M_clear, output_file = "Modulo1635_RE516171_2024_M_clear.csv")